# AutoVideoBot - Colab GPU image server (FREE)

This notebook turns Google's free T4 GPU into an image factory for the bot
running on your own PC.

## How to use it
1. **Runtime -> Change runtime type -> T4 GPU** (do this first!)
2. **Runtime -> Run all**
3. Wait for the last cell to print a link like `https://xxxx.ngrok-free.app`
4. Copy that link into the bot's `.env` file:
   ```
   IMAGE_ENDPOINT_URL=https://xxxx.ngrok-free.app
   ```
   ...or pass it for one run:  `python main.py run myvideo --topic "..." --endpoint https://xxxx`
5. Leave this notebook tab OPEN while the bot works. It pings every 25 s so
   Colab does not put the runtime to sleep.

> Links die when the notebook disconnects (~90 min idle, 12 h max). Just
> run all again and paste the new link - nothing else changes.


## 1. Install the packages (about 2 minutes)

In [ ]:
!pip install -q diffusers transformers accelerate safetensors pillow fastapi "uvicorn[standard]" sentencepiece protobuf pyngrok nest_asyncio
print("packages installed")


## 2. Check the GPU

In [ ]:
import torch
assert torch.cuda.is_available(), (
    "NO GPU! Go to Runtime -> Change runtime type -> T4 GPU, then Run all again."
)
print("GPU :", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GB")


## 3. Load the model
Change `MODEL_ID` if you like. Good picks:
* `stabilityai/stable-diffusion-xl-base-1.0` - reliable, great detail (default)
* `stabilityai/sdxl-turbo` - **4x faster**, slightly less detail, ideal for quick drafts
* `black-forest-labs/FLUX.1-schnell` - beautiful, but a big download per session


In [ ]:
import os, time
import torch
from diffusers import AutoPipelineForText2Image

MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"   # or "stabilityai/sdxl-turbo"
MAX_INTERNAL_PIXELS = 1024 * 1024     # generate at <= 1 mp, upscale after (saves VRAM)
DEFAULT_STEPS = 30
IS_TURBO = "turbo" in MODEL_ID.lower()
if IS_TURBO:
    DEFAULT_STEPS = 4

print("loading", MODEL_ID, "(first time downloads several GB) ...")
t0 = time.time()
pipe = AutoPipelineForText2Image.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
pipe = pipe.to("cuda")
pipe.enable_attention_slicing()
pipe.enable_vae_slicing()
pipe.set_progress_bar_config(disable=True)
print(f"ready in {time.time()-t0:.0f}s")


## 4. Start the server and print the public link

In [ ]:
import base64, io, threading
import nest_asyncio
from fastapi import FastAPI
from fastapi.responses import JSONResponse
from pydantic import BaseModel
import uvicorn

nest_asyncio.apply()
app = FastAPI()

class Job(BaseModel):
    id: str | None = None
    prompt: str
    negative_prompt: str | None = ""
    width: int | None = 1024
    height: int | None = 1024
    steps: int | None = None
    guidance_scale: float | None = 7.0
    seed: int | None = None

class JobBatch(BaseModel):
    jobs: list[Job]

@app.get("/health")
def health():
    return {"ok": True, "gpu": torch.cuda.get_device_name(0), "model": MODEL_ID}

def fit(w, h):
    w, h = max(256, int(w)), max(256, int(h))
    if w * h > MAX_INTERNAL_PIXELS:
        k = (MAX_INTERNAL_PIXELS / (w * h)) ** 0.5
        w, h = int(w * k), int(h * k)
    return max(256, w - w % 8), max(256, h - h % 8)

@app.post("/generate")
def generate(batch: JobBatch):
    results = []
    for job in batch.jobs:
        try:
            tw, th = max(8, round(job.width / 8) * 8), max(8, round(job.height / 8) * 8)
            gw, gh = fit(tw, th)
            steps = min(job.steps or DEFAULT_STEPS, 4) if IS_TURBO else (job.steps or DEFAULT_STEPS)
            guidance = 0.0 if IS_TURBO else float(job.guidance_scale or 7.0)
            gen = None
            if job.seed is not None:
                gen = torch.Generator(device="cuda").manual_seed(int(job.seed))
            with torch.inference_mode():
                out = pipe(prompt=job.prompt, negative_prompt=job.negative_prompt or None,
                           width=gw, height=gh, num_inference_steps=max(1, steps),
                           guidance_scale=guidance, generator=gen)
            img = out.images[0]
            if (img.width, img.height) != (tw, th):
                img = img.resize((tw, th), 1)
            buf = io.BytesIO()
            img.convert("RGB").save(buf, format="JPEG", quality=94)
            results.append({"id": job.id, "ok": True,
                            "image_b64": base64.b64encode(buf.getvalue()).decode()})
            print(f"ok   {job.id}  {gw}x{gh} -> {tw}x{th}")
        except Exception as e:
            print(f"FAIL {job.id}: {e}")
            results.append({"id": job.id, "ok": False, "error": str(e)[:300]})
        torch.cuda.empty_cache()
    return JSONResponse({"results": results})

threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000,
                                            log_level="warning"), daemon=True).start()
import time; time.sleep(3)
print("local server is up")


### 4b. Expose it to the internet with ngrok
Free ngrok accounts need a token (one-time, 2 minutes):
1. sign up at https://dashboard.ngrok.com/get-started/your-authtoken
2. paste the token below and run the cell


In [ ]:
from pyngrok import ngrok, conf

NGROK_TOKEN = ""      # <-- paste your ngrok authtoken here
if NGROK_TOKEN:
    conf.get_default().auth_token = NGROK_TOKEN

try:
    tunnel = ngrok.connect(8000)
except Exception as e:
    print("ngrok failed:", e)
    print("Falling back to a cloudflared quick tunnel (no account needed) ...")
    !wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x cloudflared
    import subprocess, re
    proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    tunnel = None
    for line in proc.stdout:
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if m:
            tunnel = type("T", (), {"public_url": m.group(0)})()
            break
    if tunnel is None:
        raise SystemExit("could not get a public link")

URL = tunnel.public_url
print("=" * 68)
print("  YOUR IMAGE SERVER IS LIVE")
print("=" * 68)
print(f"  {URL}")
print()
print("  Put this line in the bot's .env file:")
print(f"  IMAGE_ENDPOINT_URL={URL}")
print()
print("  ...or use it for a single run:")
print(f"  python main.py run myvideo --topic \"your topic\" --endpoint {URL}")
print("=" * 68)
print("  Keep this tab open while the bot works.")


## 5. Test it by hand (optional)
Generates one image and shows it, so you know the GPU works before
starting a long video build.

In [ ]:
import requests
r = requests.post(URL + "/generate", json={"jobs": [{
    "id": "test", "prompt": "a black hole bending light, cinematic",
    "width": 1280, "height": 720, "steps": 20, "guidance_scale": 7.0}]}, timeout=600)
res = r.json()["results"][0]
print("ok" if res.get("ok") else res)
if res.get("ok"):
    import base64, io
    from PIL import Image
    img = Image.open(io.BytesIO(base64.b64decode(res["image_b64"])))
    img.save("/content/test.jpg")
    display(img)


## Troubleshooting
| Symptom | Fix |
|---|---|
| `assert NO GPU` | Runtime -> Change runtime type -> **T4 GPU**, then Run all |
| Out of memory | Lower `MAX_INTERNAL_PIXELS` to `512*512`, or use `sdxl-turbo` |
| ngrok refuses to connect | Paste your authtoken, or use the cloudflared fallback |
| The bot says "Colab unavailable" | Re-run this notebook: the link expired. Paste the new one in `.env` |
| Generation is slow (60 s/image) | You are on a free T4 - expected. Use `sdxl-turbo` for drafts |

**Free tier limits:** ~90 minutes idle timeout, 12 hours maximum session,
and a daily GPU quota. For long videos, run the bot in batches with
`--only s01-s10`, then `--only s11-s20`, and so on.
